# 03 — PaySim leakage audit and positive control

This notebook classifies PaySim columns by decision-time availability and contrasts strict prior-history aggregates with deliberately leaky full-history aggregates.

The leaky path is a **positive control**, never a deployable feature pipeline.

In [1]:
from pathlib import Path

from IPython.display import Markdown, display

from pit_fintech.data.paysim import (
    BALANCE_COLUMNS,
    DATASET_URL,
    LABEL_COLUMNS,
    POLICY_OUTPUT_COLUMNS,
    REQUEST_TIME_COLUMNS,
    connect_paysim,
    find_paysim_csv,
    resolve_project_root,
    setup_instructions,
)

PROJECT_ROOT = resolve_project_root(Path.cwd())
paysim_csv = find_paysim_csv(PROJECT_ROOT)
DATA_AVAILABLE = paysim_csv is not None

if DATA_AVAILABLE:
    connection = connect_paysim(paysim_csv)
    print({"path": str(paysim_csv), "status": "ready"})
else:
    connection = None
    display(Markdown("```text\n" + setup_instructions(PROJECT_ROOT) + "\n```"))


def show(sql: str):
    if connection is None:
        print("Query skipped: PaySim CSV is unavailable.")
        return None
    table = connection.sql(sql).to_arrow_table()
    display(table)

{'path': 'C:\\workspace\\pit-fintech\\data\\raw\\PS_20174392719_1491204439457_log.csv', 'status': 'ready'}


## Feature-availability policy

The Kaggle dataset page explicitly warns that fraudulent transactions are cancelled and that the four balance columns should not be used for fraud detection. This project therefore excludes them from the deployable baseline rather than treating them as ordinary request fields.

In [2]:
feature_policy = {
    "request-time context": REQUEST_TIME_COLUMNS,
    "excluded balance/post-outcome risk": BALANCE_COLUMNS,
    "label only": LABEL_COLUMNS,
    "existing policy output; excluded from baseline": POLICY_OUTPUT_COLUMNS,
    "derived historical features": (
        "origin_txn_count_<window>",
        "origin_amount_sum_<window>",
        "origin_unique_destinations_<window>",
        "destination_received_count_<window>",
        "time_since_origin_last_txn",
    ),
}
display(feature_policy)
print("Source policy:", DATASET_URL)

{'request-time context': ('step', 'type', 'amount', 'nameOrig', 'nameDest'),
 'excluded balance/post-outcome risk': ('oldbalanceOrg',
  'newbalanceOrig',
  'oldbalanceDest',
  'newbalanceDest'),
 'label only': ('isFraud',),
 'existing policy output; excluded from baseline': ('isFlaggedFraud',),
 'derived historical features': ('origin_txn_count_<window>',
  'origin_amount_sum_<window>',
  'origin_unique_destinations_<window>',
  'destination_received_count_<window>',
  'time_since_origin_last_txn')}

Source policy: https://www.kaggle.com/datasets/ealaxi/paysim1


## Inspect the excluded fields

This query is diagnostic only. A strong relationship with the label is not permission to use a field; availability at the decision cutoff is the gate.

In [3]:
show(
    """
    SELECT
        isFraud,
        count(*) AS rows,
        round(avg(oldbalanceOrg), 2) AS mean_old_origin_balance,
        round(avg(newbalanceOrig), 2) AS mean_new_origin_balance,
        round(avg(oldbalanceDest), 2) AS mean_old_destination_balance,
        round(avg(newbalanceDest), 2) AS mean_new_destination_balance,
        round(100.0 * avg((oldbalanceOrg = newbalanceOrig)::INTEGER), 4)
            AS unchanged_origin_balance_percent
    FROM paysim
    GROUP BY isFraud
    ORDER BY isFraud
    """
)

pyarrow.Table
isFraud: int32
rows: int64
mean_old_origin_balance: double
mean_new_origin_balance: double
mean_old_destination_balance: double
mean_new_destination_balance: double
unchanged_origin_balance_percent: double
----
isFraud: [[0,1]]
rows: [[6354407,8213]]
mean_old_origin_balance: [[832828.71,1649667.61]]
mean_new_origin_balance: [[855970.23,192392.63]]
mean_old_destination_balance: [[1101420.87,544249.62]]
mean_new_destination_balance: [[1224925.68,1279707.62]]
unchanged_origin_balance_percent: [[32.8745,0.694]]

In [4]:
show(
    """
    SELECT
        isFlaggedFraud,
        count(*) AS rows,
        sum(isFraud) AS fraud_rows,
        round(100.0 * avg(isFraud), 6) AS fraud_percent,
        min(amount) AS min_amount,
        max(amount) AS max_amount
    FROM paysim
    GROUP BY isFlaggedFraud
    ORDER BY isFlaggedFraud
    """
)

pyarrow.Table
isFlaggedFraud: int32
rows: int64
fraud_rows: decimal128(38, 0)
fraud_percent: double
min_amount: double
max_amount: double
----
isFlaggedFraud: [[0,1]]
rows: [[6362604,16]]
fraud_rows: [[8197,16]]
fraud_percent: [[0.128831,100]]
min_amount: [[0,353874.22]]
max_amount: [[92445516.64,10000000]]

## Positive control: strict prior history versus full history

To keep the notebook local-friendly, the comparison uses the 500 most repeated origin entities. `pit_prior_amount` contains only rows ordered before the current transaction. `leaky_full_history_amount` includes the current row and future rows. This cumulative prototype demonstrates the cutoff failure; production feature windows will be defined separately in `FeatureSpec`.

In [5]:
show(
    """
    WITH selected_entities AS (
        SELECT nameOrig
        FROM paysim
        GROUP BY nameOrig
        HAVING count(*) >= 3
        ORDER BY count(*) DESC, nameOrig
        LIMIT 500
    ), scoped AS (
        SELECT p.*
        FROM paysim p
        SEMI JOIN selected_entities e USING (nameOrig)
    ), compared AS (
        SELECT
            source_row_number,
            step,
            nameOrig,
            amount,
            coalesce(
                sum(amount) OVER (
                    PARTITION BY nameOrig
                    ORDER BY step, source_row_number
                    ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
                ),
                0.0
            ) AS pit_prior_amount,
            sum(amount) OVER (PARTITION BY nameOrig) AS leaky_full_history_amount
        FROM scoped
    )
    SELECT
        count(*) AS compared_rows,
        count_if(abs(pit_prior_amount - leaky_full_history_amount) > 1e-9)
            AS rows_changed_by_future_or_current_data,
        round(
            100.0 * count_if(abs(pit_prior_amount - leaky_full_history_amount) > 1e-9)
            / count(*),
            4
        ) AS changed_percent,
        round(avg(leaky_full_history_amount - pit_prior_amount), 2)
            AS mean_leaked_amount
    FROM compared
    """
)

pyarrow.Table
compared_rows: int64
rows_changed_by_future_or_current_data: decimal128(38, 0)
changed_percent: double
mean_leaked_amount: double
----
compared_rows: [[45]]
rows_changed_by_future_or_current_data: [[45]]
changed_percent: [[100]]
mean_leaked_amount: [[355654.66]]

In [6]:
show(
    """
    WITH chosen AS (
        SELECT nameOrig
        FROM paysim
        GROUP BY nameOrig
        HAVING count(*) >= 5
        ORDER BY count(*) DESC, nameOrig
        LIMIT 1
    )
    SELECT
        source_row_number,
        step,
        nameOrig,
        amount,
        coalesce(
            sum(amount) OVER (
                PARTITION BY nameOrig
                ORDER BY step, source_row_number
                ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
            ),
            0.0
        ) AS pit_prior_amount,
        sum(amount) OVER (PARTITION BY nameOrig) AS leaky_full_history_amount
    FROM paysim
    SEMI JOIN chosen USING (nameOrig)
    ORDER BY step, source_row_number
    """
)

pyarrow.Table
source_row_number: int64
step: int64
nameOrig: string
amount: double
pit_prior_amount: double
leaky_full_history_amount: double
----
source_row_number: []
step: []
nameOrig: []
amount: []
pit_prior_amount: []
leaky_full_history_amount: []

## Leakage gate for the next implementation step

- Keep `isFraud` in a separate label table after feature construction.
- Exclude all four balance columns from the deployable baseline because the dataset source documents cancellation/post-outcome risk.
- Exclude `isFlaggedFraud` from the baseline because it is an existing policy output, not neutral transaction context.
- Permit `type` and `amount` as current-request fields, but never let transaction `t` update historical aggregates before scoring `t`.
- Derive history only from `(step, source_row_number)` values strictly preceding the cutoff.
- Keep the full-history aggregate only as experiment E2, the deliberately leaky positive control.